# CortexHire sandbox demo

Reviewer-facing sandbox for the Redrob hackathon submission. It clones the public repo, installs dependencies, runs the same deterministic `rank.py` entry point on a small sample, validates the output shape, previews the ranking, and exposes the generated CSV for download.

The full 100K organizer dataset is private and not bundled here. Stage 3 full reproduction uses the repo command documented in `README.md` with the organizer data and shipped artifacts.


## 1. Setup

Run all cells from a fresh Colab runtime. Package installation uses the network; the ranking command itself is CPU-only and does not call hosted APIs.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/durvishkhurana/CortexHire.git"
REPO_DIR = Path("CortexHire")

if Path.cwd().name != REPO_DIR.name:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir(REPO_DIR)

print("repo:", Path.cwd())
print("python:", sys.version.split()[0])


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("dependencies installed")


## 2. Choose input

Default: use the committed synthetic fixture. To test another small JSON/JSONL candidate file, set `USE_UPLOAD = True` and rerun this cell.


In [ ]:
from pathlib import Path

USE_UPLOAD = False
candidate_path = Path("tests/fixtures/synthetic_candidates.jsonl")

if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        candidate_path = Path(next(iter(uploaded)))

assert candidate_path.exists(), candidate_path
print("candidate input:", candidate_path)


## 3. Run deterministic ranking

This uses the same CLI as the final system. The sandbox intentionally points `--artifacts` to an empty directory so the repo's built-in baseline path is exercised without private artifacts.


In [ ]:
import time

out_path = Path("sandbox_submission.csv")
cmd = [
    sys.executable,
    "rank.py",
    "--candidates",
    str(candidate_path),
    "--out",
    str(out_path),
    "--artifacts",
    "/tmp/no_artifacts",
]

start = time.perf_counter()
result = subprocess.run(cmd, text=True, capture_output=True)
elapsed = time.perf_counter() - start

print("command:", " ".join(cmd))
print(f"elapsed_seconds={elapsed:.2f}")
print(result.stdout)
print(result.stderr)
assert result.returncode == 0, result.returncode
assert out_path.exists(), out_path


## 4. Validate and preview

The real submission must contain exactly 100 rows. This small sample contains fewer candidates, so the checks below validate the same column contract, rank order, score order, uniqueness, and reasoning presence.


In [ ]:
import pandas as pd
from IPython.display import display

df = pd.read_csv(out_path)
expected_cols = ["candidate_id", "rank", "score", "reasoning"]

checks = {
    "columns_ok": list(df.columns) == expected_cols,
    "unique_candidate_ids": df["candidate_id"].is_unique,
    "unique_ranks": df["rank"].is_unique,
    "ranks_start_at_1": int(df["rank"].min()) == 1,
    "scores_non_increasing": df["score"].is_monotonic_decreasing,
    "score_min_ge_0": float(df["score"].min()) >= 0.0,
    "score_max_le_100": float(df["score"].max()) <= 100.0,
    "reasoning_present": df["reasoning"].fillna("").str.len().gt(0).all(),
}

display(pd.DataFrame([checks]).T.rename(columns={0: "pass"}))
assert all(checks.values()), checks

print(f"rows={len(df)}")
print(f"score_range={df['score'].min():.6f}..{df['score'].max():.6f}")
display(df)


## 5. Score curve

A quick visual check that the output is ranked and scores are monotonic.


In [ ]:
import matplotlib.pyplot as plt

ax = df.plot(x="rank", y="score", marker="o", legend=False, figsize=(8, 4))
ax.set_title("Sandbox ranking score by rank")
ax.set_xlabel("rank")
ax.set_ylabel("score")
ax.grid(True, alpha=0.3)
plt.show()


## 6. Download output

Optional convenience cell for reviewers who want the generated CSV locally.


In [ ]:
DOWNLOAD = False

if DOWNLOAD:
    from google.colab import files
    files.download(str(out_path))
else:
    print("CSV ready at", out_path.resolve())


## Full reproduction command

When the private organizer data and artifacts are present, the final submission is reproduced with:

```bash
python rank.py --candidates ./data/candidates.jsonl --out ./submission.csv --artifacts ./artifacts
python validate_submission.py submission.csv
```
